# Week 9 Lab: Power Analysis and Endogeneity Bias

## Learning Objectives

By the end of this lab, you will be able to:
1. Understand the concepts of size and power of hypothesis tests
2. Implement power simulations via Monte Carlo
3. Analyze how endogeneity affects OLS estimation and inference
4. Use closures to create flexible simulation functions

## Recap: The Schooling DGP

$$\begin{align}
u &\sim N(0, \sigma_u^2) \\
A &\sim N(0, \sigma_A^2) && \text{(unobserved ability)} \\
S &= \pi + A && \text{(schooling)} \\
Y &= \beta_1 + \beta_2 S + \beta_3 A + u && \text{(log earnings)}
\end{align}$$

**Key insight**: When $\beta_3 \neq 0$, ability affects earnings directly. Since ability also affects schooling, this creates **endogeneity**: $\text{Cov}(S, \varepsilon) \neq 0$ where $\varepsilon = \beta_3 A + u$.

In [ ]:
using Distributions, Random, Plots, Statistics, LinearAlgebra

# Load functions from earlier weeks (provides lm_ols, lm_inference, etc.)
# Note: For the power simulations below, we use manual OLS with homoskedastic SEs
# to illustrate the theoretical concepts. The lm_ols function provides robust SEs.
include("emet_8014_functions.jl")

## Exercise 1: Augmenting the DGP Function

Last week you wrote the function `schooling_sample` which had two arguments:

* true coefficient $\beta_2$ and sample size $N$

We shut off the effect of ability on earnings by setting $\beta_3=0$.

Today we want to allow for $\beta_3$ to take nonzero values, which is the source of endogeneity in this model. Technically, we do this by adding $\beta_3$ as a third argument into the `schooling_sample` function. 

Here's the scaffolding for you, please fill in the details:

(notice that we're hardwiring the keyword arguments to the values that we calibrated last week. For example, the calibrated value for $\pi$ was 13.2.)

In [ ]:
"""
    schooling_sample(β2, β3, N; π=13.2, β1=4.7, σ2_u=0.175, σ2_A=7.2)

Generate a random sample from the schooling DGP.

# Arguments (positional)
- `β2`: Return to schooling (parameter of interest)
- `β3`: Effect of ability on earnings (controls endogeneity)
- `N`: Sample size

# Keyword arguments (fixed parameters)
- `π`: Mean schooling
- `β1`: Intercept
- `σ2_u`: Variance of idiosyncratic error
- `σ2_A`: Variance of ability

# Returns
- `S`: Vector of schooling values
- `Y`: Vector of log earnings
"""
function schooling_sample(β2::Real, β3::Real, N::Integer;
                          π::Real=13.2, β1::Real=4.7, 
                          σ2_u::Real=0.175, σ2_A::Real=7.2)
    # Draw random components
    # Hint: Use rand(Normal(mean, std), N) - remember Normal takes std, not variance!
    u = nothing  # YOUR CODE HERE
    A = nothing  # YOUR CODE HERE
    
    # Generate observables
    S = nothing  # YOUR CODE HERE
    Y = nothing  # YOUR CODE HERE
    
    return S, Y
end

## Exercise 2: Creating a Closure

A **closure** is a function that "remembers" variables from where it was created. Here's a simple example:
```julia
function make_adder(a)
    return x -> x + a   # This inner function "captures" a
end

add5 = make_adder(5)
add5(10)  # Returns 15 — the function "remembers" that a = 5
```

The inner function `x -> x + a` doesn't just see `a` at creation time—it *holds onto* that value even after `make_adder` returns.

**Why is this useful here?** Our DGP has two kinds of parameters:
- **Calibration parameters** (π, β₁, σ²ᵤ, σ²ₐ): Fixed across all simulations
- **Experimental parameters** (β₂, β₃, N): What we vary to study power

Rather than passing all 7 arguments every time, we'll create a closure that "bakes in" the calibration, giving us a cleaner interface:
```julia
dgp = make_dgp(13.2, 4.7, 0.175, 7.2)  # Fix calibration once
S, Y = dgp(0.07, 0.0, 100)              # Now just vary what matters
```

We'll create a closure that fixes the calibrated parameters, leaving only $(\beta_2, \beta_3, n)$ as arguments.

In [ ]:
"""
    make_dgp(π, β1, σ2_u, σ2_A)

Create a DGP closure with fixed calibration parameters.

Returns a function `dgp(β2, β3, N)` that generates schooling samples.

# Example
```julia
dgp = make_dgp(13.2, 4.7, 0.175, 7.2)
S, Y = dgp(0.07, 0.0, 100)  # Generate sample with β2=0.07, β3=0, N=100
```
"""
function make_dgp(π::Real, β1::Real, σ2_u::Real, σ2_A::Real)
    # Return a function that "closes over" the calibration parameters
    # Hint: Use an anonymous function (β2, β3, N) -> ...
    # YOUR CODE HERE
    error("Not yet implemented")
end

# Create our baseline DGP with calibrated parameters
dgp_base = make_dgp(13.2, 4.7, 0.175, 7.2)

In [ ]:
# Test the closure
Random.seed!(42)
S, Y = dgp_base(0.07, 0.0, 30)  # β2 = 0.07, β3 = 0 (no endogeneity), N = 30

println("Generated sample with closure:")
println("  Sample size: $(length(S))")
println("  Mean schooling: $(round(mean(S), digits=2))")

## Statistical Power: Concepts

### Hypothesis Testing Review

Consider testing $H_0: \beta_2 = \beta_2^{\text{null}}$ vs $H_1: \beta_2 \neq \beta_2^{\text{null}}$ at significance level $\alpha = 0.05$.

We compute the t-statistic:
$$t = \frac{\hat{\beta}_2 - \beta_2^{\text{null}}}{\text{SE}(\hat{\beta}_2)}$$

and reject $H_0$ if $|t| > 1.96$.

### Size vs Power

- **Size**: $P(\text{reject } H_0 | H_0 \text{ true}) = \alpha$ (ideally 0.05)
  - This is the probability of a **Type I error** (false positive)
  
- **Power**: $P(\text{reject } H_0 | H_0 \text{ false})$
  - This is the probability of correctly detecting a false null hypothesis
  - Higher power is better!

### What Affects Power?

1. **Effect size**: $|\beta_2 - \beta_2^{\text{null}}|$ (bigger effect -> higher power)
2. **Sample size**: $N$ (more data -> higher power)
3. **Variance**: $\sigma^2_u / \sigma^2_A$ (less noise -> higher power)
4. **Significance level**: $\alpha$ (larger alpha -> higher power but more false positives)

## Exercise 3: Power Simulation Function

We simulate the power by:
1. Generating many samples under the true DGP
2. Testing $H_0: \beta_2 = \beta_2^{\text{null}}$ for each sample
3. Computing the proportion of rejections

In [ ]:
"""
    schooling_power(β2, β3, N, dgp, β2_null; R=1000, α=0.05)

Simulate the power of testing H0: β2 = β2_null via Monte Carlo.

# Arguments
- `β2`: True value of β2 in the DGP
- `β3`: True value of β3 in the DGP (controls endogeneity)
- `N`: Sample size
- `dgp`: DGP function (closure) that takes (β2, β3, N)
- `β2_null`: Value of β2 under the null hypothesis
- `R`: Number of Monte Carlo replications
- `α`: Significance level (default 0.05)

# Returns
- Empirical power (proportion of replications that reject H0)

# Notes
When β2 = β2_null (null is true), this returns the empirical size.
We use homoskedastic standard errors: SE = sqrt(s2_u / (N * Var(S)))
"""
function schooling_power(β2::Real, β3::Real, N::Integer, 
                         dgp::Function, β2_null::Real;
                         R::Integer=1000, α::Real=0.05)
    
    # Critical value for two-sided test
    # Hint: Use quantile(Normal(), 1 - α/2) to get 1.96 for α = 0.05
    z_crit = nothing  # YOUR CODE HERE
    
    n_rejections = 0
    
    for r in 1:R
        Random.seed!(r)  # Different seed for each replication
        
        # Generate sample
        S, Y = dgp(β2, β3, N)
        
        # OLS estimation
        X = hcat(ones(N), S)
        β_hat = X \ Y
        
        # Residuals and variance estimate
        u_hat = Y - X * β_hat
        s2_u = sum(u_hat.^2) / N
        
        # Standard error of β_hat2 (homoskedastic formula)
        # SE = sqrt(s2_u / (N * Var(S)))
        s2_S = var(S)
        se_β2 = nothing  # YOUR CODE HERE
        
        # t-statistic: |β_hat2 - β2_null| / SE
        t_stat = nothing  # YOUR CODE HERE
        
        # Count rejections (reject if |t| > z_crit)
        if t_stat > z_crit
            n_rejections += 1
        end
    end
    
    return n_rejections / R
end

In [ ]:
# Test: Empirical size (when null is true)
# Should be close to 0.05
empirical_size = schooling_power(0.07, 0.0, 1000, dgp_base, 0.07)
println("Empirical size (testing true null): $(round(empirical_size, digits=3))")
println("(Should be close to 0.05)")

## Exercise 4: Power Table

Let's compute power for various configurations with $\beta_3 = 0$ (no endogeneity).

In [ ]:
# Power calculations for different scenarios
β2_true = 0.07
β3_true = 0.0  # No endogeneity

println("Power Table (β2 = 0.07, β3 = 0, no endogeneity)")
println("="^60)
println()

for N in [30, 100, 1000, 10_000]
    println("Sample size N = $N:")
    println("-"^40)
    
    for β2_null in [0.07, 0.05, 0.09]
        power = schooling_power(β2_true, β3_true, N, dgp_base, β2_null)
        
        if β2_null == β2_true
            println("  H0: β2 = $β2_null (TRUE)  -> Size = $(round(power, digits=3))")
        else
            println("  H0: β2 = $β2_null (FALSE) -> Power = $(round(power, digits=3))")
        end
    end
    println()
end

### Interpreting the Results

1. **Size**: When $\beta_2^{\text{null}} = 0.07$ (true null), rejection rate should be approximately 0.05

2. **Power increases with n**: Larger samples make it easier to detect false nulls

3. **Power increases with effect size**: Testing $\beta_2 = 0.05$ vs $\beta_2 = 0.09$ both have $|\beta_2 - 0.07| = 0.02$, so similar power

## Exercise 5: Power Curve

A **power curve** shows power as a function of the null hypothesis value.

In [ ]:
"""
    plot_power_curve(β2, β3, N, dgp; β2_range=0.05, n_points=41, R=1000)

Plot the power curve for testing H0: β2 = β2_null.

# Arguments
- `β2`: True value of β2
- `β3`: True value of β3 (controls endogeneity)
- `N`: Sample size
- `dgp`: DGP function
- `β2_range`: Half-width of null hypothesis range to test
- `n_points`: Number of null values to evaluate
- `R`: Monte Carlo replications per point

# Returns
- Plot object showing the power curve
"""
function plot_power_curve(β2::Real, β3::Real, N::Integer, dgp::Function;
                          β2_range::Real=0.05, n_points::Integer=21, R::Integer=500)
    
    # Grid of null hypothesis values
    β2_null_grid = range(β2 - β2_range, β2 + β2_range, length=n_points)
    
    # Compute power at each point
    # YOUR CODE HERE: Use a comprehension to compute power for each β2_null
    # Hint: [schooling_power(...) for β2_null in β2_null_grid]
    powers = nothing
    
    # Create plot
    p = plot(β2_null_grid, powers,
             xlabel="β2 under H0",
             ylabel="Power",
             title="Power Curve (N = $N, β2 = $β2, β3 = $β3)",
             legend=:topright,
             label="Power",
             linewidth=2,
             marker=:circle,
             markersize=3)
    
    # Add reference lines
    vline!(p, [β2], label="True β2", linestyle=:dash, color=:red, linewidth=2)
    hline!(p, [0.05], label="α = 0.05", linestyle=:dot, color=:gray)
    
    return p
end

In [ ]:
# Power curve with no endogeneity
plot_power_curve(0.07, 0.0, 100, dgp_base; R=500)

In [ ]:
# Compare power curves for different sample sizes
plots = [plot_power_curve(0.07, 0.0, N, dgp_base; R=300) for N in [30, 100, 500, 1000]]
plot(plots..., layout=(2, 2), size=(900, 700))

## Exercise 6: Effect of Endogeneity on Power

Now let's see what happens when $\beta_3 \neq 0$ (endogeneity exists).

### The Probability Limit Under Endogeneity

When the econometrician estimates $Y_i = \beta_1 + \beta_2 S_i + \varepsilon_i$ but the true model has $\varepsilon_i = \beta_3 A_i + u_i$:

$$\text{plim } \hat{\beta}_2^{OLS} = \beta_2 + \beta_3 \cdot \frac{\text{Cov}(S, A)}{\text{Var}(S)}$$

Since $S = \pi + A$, we have $\text{Cov}(S, A) = \text{Var}(A) = \sigma^2_A$ and $\text{Var}(S) = \sigma^2_A$.

Therefore: $\text{plim } \hat{\beta}_2^{OLS} = \beta_2 + \beta_3$

**The OLS estimator is biased by $\beta_3$!**

In [ ]:
# Power curve WITH endogeneity
# β3 = 0.03 means OLS will converge to β2 + β3 = 0.07 + 0.03 = 0.10

plot_power_curve(0.07, 0.03, 1000, dgp_base; R=500)

### What Happened?

With $\beta_3 = 0.03$, the power curve **shifts to the right** by 0.03!

- The minimum is now at $\beta_2^{\text{null}} = 0.10$ (not 0.07)
- Testing $H_0: \beta_2 = 0.07$ now has high power (even though 0.07 is the true value!)

**This is disastrous for inference:**
- We'd reject the true null hypothesis with high probability
- Our confidence intervals would not contain the true value
- This is why endogeneity is such a serious problem in econometrics

In [ ]:
# Compare: no endogeneity vs endogeneity
p1 = plot_power_curve(0.07, 0.00, 1000, dgp_base; R=300)
title!(p1, "No Endogeneity (β3 = 0)")

p2 = plot_power_curve(0.07, 0.03, 1000, dgp_base; R=300)
title!(p2, "With Endogeneity (β3 = 0.03)")

plot(p1, p2, layout=(1, 2), size=(1000, 400))

## Summary

### Key Concepts

1. **Size**: Probability of rejecting a true null (should equal $\alpha$)
2. **Power**: Probability of rejecting a false null (higher is better)
3. **Power increases with**: sample size, effect size, lower variance

### Closures in Julia

- Closures "capture" variables from their enclosing scope
- Useful for creating specialized functions with fixed parameters
- Example: `dgp_base = make_dgp(π, β1, σ2_u, σ2_A)`

### The Endogeneity Problem

When $\beta_3 \neq 0$:
- OLS converges to $\beta_2 + \beta_3$, not $\beta_2$
- The power curve shifts by $\beta_3$
- Hypothesis tests have incorrect size (we reject true nulls)
- Confidence intervals miss the true value

**Solution**: Use instrumental variables or find natural experiments.

### Looking Ahead

In Week 10, we'll return to Card's data and implement:
- Complete IV estimation with standard errors
- 2SLS with multiple instruments
- First-stage F-tests for weak instruments